In [1]:
# ============================================================
# R1_ROMA_rebuild_regime_signals_leakage_controlled.ipynb
# ROMA-TWETF Clean Rebuild: Leakage-Controlled Regime Signals
#
# Purpose:
# 1. Rebuild ROMA from scratch using the leakage-controlled AURORA dataset.
# 2. Use the same ETF universe and target definitions as AURORA.
# 3. Train ROMA-style regime models under purged walk-forward splits.
# 4. Generate out-of-sample regime probabilities and predictions.
# 5. Select validation-supported ROMA probability sources for R2 allocation.
#
# This notebook creates:
# outputs/ROMA_TWETF/purged_walk_forward_regime_signals/run_<RUN_ID>/
#   metrics/
#   probabilities/
#   predictions/
#   tables/
#   reports/
#   NOTEBOOK_R2_ROMA_INPUT_INDEX.csv
#
# Educational/research use only.
# Not personalized financial advice.
# ============================================================

from __future__ import annotations

import json
import math
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    cohen_kappa_score,
    log_loss,
    confusion_matrix,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except Exception:
    HAS_LIGHTGBM = False
    print("LightGBM not available. ROMA_M4_LightGBM will be skipped.")

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False
    print("XGBoost not available. ROMA_M5_XGBoost will be skipped.")

# ============================================================
# 1. Paths and run configuration
# ============================================================

PROJECT_CODE = "ROMA_TWETF"
SHARED_AURORA_CODE = "AURORA_TWETF"

PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
MODELING_DIR = DATA_ROOT / "modeling"
PANEL_DIR = DATA_ROOT / "panels"

AURORA_FEATURE_PATH = MODELING_DIR / "AURORA_TWETF_features_with_labels.parquet"
AURORA_FEATURE_CSV_PATH = MODELING_DIR / "AURORA_TWETF_features_with_labels.csv"

AURORA_ETF_RETURN_PANEL_PATH = PANEL_DIR / "AURORA_etf_return_panel.parquet"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
GLOBAL_OUTPUT_ROOT = PUBLICATION_ROOT / "outputs"

TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
FIGURE_DIR = OUTPUT_ROOT / "figures"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "purged_walk_forward_regime_signals" / f"run_{RUN_ID}"

METRIC_DIR = RUN_ROOT / "metrics"
PROBA_DIR = RUN_ROOT / "probabilities"
PRED_DIR = RUN_ROOT / "predictions"
TABLE_RUN_DIR = RUN_ROOT / "tables"
REPORT_RUN_DIR = RUN_ROOT / "reports"
MODEL_INFO_DIR = RUN_ROOT / "model_info"

for d in [
    OUTPUT_ROOT,
    TABLE_DIR,
    REPORT_DIR,
    FIGURE_DIR,
    RUN_ROOT,
    METRIC_DIR,
    PROBA_DIR,
    PRED_DIR,
    TABLE_RUN_DIR,
    REPORT_RUN_DIR,
    MODEL_INFO_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("ROMA-TWETF R1: Clean Leakage-Controlled Regime Signal Rebuild")
print("=" * 80)
print("Timestamp UTC       :", RUN_TIMESTAMP)
print("Run ID              :", RUN_ID)
print("AURORA feature path :", AURORA_FEATURE_PATH)
print("ETF return panel    :", AURORA_ETF_RETURN_PANEL_PATH)
print("Run root            :", RUN_ROOT)
print("=" * 80)

if not AURORA_FEATURE_PATH.exists() and not AURORA_FEATURE_CSV_PATH.exists():
    raise FileNotFoundError(
        f"Could not find AURORA leakage-controlled feature file:\n"
        f"{AURORA_FEATURE_PATH}\n"
        f"{AURORA_FEATURE_CSV_PATH}"
    )

if not AURORA_ETF_RETURN_PANEL_PATH.exists():
    print("Warning: ETF return panel not found. R1 can still run, but R2 will require ETF returns.")
    print("Missing:", AURORA_ETF_RETURN_PANEL_PATH)

# ============================================================
# 2. Global settings
# ============================================================

ETF_UNIVERSE = ["0050", "006208", "00692", "00881"]
CASH_COL = "CASH"

TARGET_COLS = [
    "TAIEX_regime_fixed_20d",
    "TAIEX_regime_fixed_60d",
]

TARGET_HORIZON_MAP = {
    "TAIEX_regime_fixed_20d": 20,
    "TAIEX_regime_fixed_60d": 60,
}

CLASS_LABELS = [0, 1, 2, 3, 4]

REGIME_LABELS = {
    0: "strong_bear",
    1: "bear",
    2: "neutral",
    3: "bull",
    4: "strong_bull",
}

RANDOM_STATE = 42
N_JOBS = -1

# Same purged walk-forward philosophy as AURORA Notebook 07.
# These folds are chronological and use target-specific embargoes.
WALK_FORWARD_FOLDS = [
    {
        "fold_id": "WF1",
        "train_end_frac": 0.55,
        "val_end_frac": 0.70,
        "test_end_frac": 0.85,
    },
    {
        "fold_id": "WF2",
        "train_end_frac": 0.65,
        "val_end_frac": 0.80,
        "test_end_frac": 0.95,
    },
    {
        "fold_id": "WF3",
        "train_end_frac": 0.70,
        "val_end_frac": 0.85,
        "test_end_frac": 1.00,
    },
]

# Validation metric weights for ROMA model selection.
# ROMA is intended as a return-seeking regime allocation signal,
# but R1 only selects forecasting probability sources.
SELECTION_METRIC_WEIGHTS = {
    "macro_f1": 0.30,
    "balanced_accuracy": 0.25,
    "qwk": 0.25,
    "adjacent_accuracy": 0.10,
    "neg_ordinal_mae": 0.10,
}

# Probability ensemble settings.
ENSEMBLE_TOP_K = 4
ENSEMBLE_MIN_WEIGHT = 0.05

# ============================================================
# 3. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)

    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []

    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })

    return pd.DataFrame(rows)

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
        .replace(".", "_")
        .replace("%", "pct")
    )

def proba_cols():
    return [f"proba_class_{c}" for c in CLASS_LABELS]

def normalize_proba_array(p):
    p = np.asarray(p, dtype=float)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p[p < 0] = 0.0

    row_sums = p.sum(axis=1, keepdims=True)
    zero_rows = row_sums[:, 0] <= 0

    if np.any(zero_rows):
        p[zero_rows, :] = 1.0 / p.shape[1]
        row_sums = p.sum(axis=1, keepdims=True)

    return p / row_sums

def predict_proba_fixed_classes(model, X, classes=CLASS_LABELS):
    if hasattr(model, "predict_proba"):
        raw = model.predict_proba(X)
        model_classes = list(getattr(model, "classes_", classes))

        out = np.zeros((X.shape[0], len(classes)), dtype=float)

        for j, cls in enumerate(model_classes):
            if cls in classes:
                out[:, classes.index(cls)] = raw[:, j]

        return normalize_proba_array(out)

    pred = model.predict(X)
    out = np.zeros((len(pred), len(classes)), dtype=float)

    for i, cls in enumerate(pred):
        if cls in classes:
            out[i, classes.index(cls)] = 1.0
        else:
            out[i, :] = 1.0 / len(classes)

    return normalize_proba_array(out)

def hard_prediction_from_proba(p):
    p = normalize_proba_array(p)
    return np.asarray(CLASS_LABELS)[np.argmax(p, axis=1)]

def expected_class_from_proba(p):
    p = normalize_proba_array(p)
    return p @ np.asarray(CLASS_LABELS, dtype=float)

def ordinal_variance_from_proba(p):
    p = normalize_proba_array(p)
    cls = np.asarray(CLASS_LABELS, dtype=float)
    mean = p @ cls
    second = p @ (cls ** 2)
    return second - mean ** 2

def entropy_from_proba(p):
    p = normalize_proba_array(p)
    ent = -np.sum(np.clip(p, 1e-12, 1.0) * np.log(np.clip(p, 1e-12, 1.0)), axis=1)
    return ent

def adjacent_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(np.abs(y_true - y_pred) <= 1))

def ordinal_mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def ordinal_rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))

def multiclass_brier(y_true, proba):
    y_true = np.asarray(y_true)
    proba = normalize_proba_array(proba)
    y_onehot = np.zeros_like(proba)

    for i, cls in enumerate(y_true):
        if cls in CLASS_LABELS:
            y_onehot[i, CLASS_LABELS.index(cls)] = 1.0

    return float(np.mean(np.sum((proba - y_onehot) ** 2, axis=1)))

def expected_calibration_error(y_true, proba, n_bins=10):
    y_true = np.asarray(y_true)
    proba = normalize_proba_array(proba)

    pred = hard_prediction_from_proba(proba)
    conf = np.max(proba, axis=1)
    correct = (pred == y_true).astype(float)

    ece = 0.0
    bins = np.linspace(0.0, 1.0, n_bins + 1)

    for i in range(n_bins):
        left = bins[i]
        right = bins[i + 1]

        if i == n_bins - 1:
            mask = (conf >= left) & (conf <= right)
        else:
            mask = (conf >= left) & (conf < right)

        if mask.sum() == 0:
            continue

        bin_acc = correct[mask].mean()
        bin_conf = conf[mask].mean()
        ece += (mask.mean()) * abs(bin_acc - bin_conf)

    return float(ece)

def calculate_metrics(y_true, proba):
    y_true = np.asarray(y_true).astype(int)
    proba = normalize_proba_array(proba)
    y_pred = hard_prediction_from_proba(proba)

    out = {}

    out["accuracy"] = float(accuracy_score(y_true, y_pred))

    try:
        out["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred))
    except Exception:
        out["balanced_accuracy"] = np.nan

    try:
        out["macro_f1"] = float(f1_score(y_true, y_pred, labels=CLASS_LABELS, average="macro", zero_division=0))
    except Exception:
        out["macro_f1"] = np.nan

    try:
        out["weighted_f1"] = float(f1_score(y_true, y_pred, labels=CLASS_LABELS, average="weighted", zero_division=0))
    except Exception:
        out["weighted_f1"] = np.nan

    try:
        out["qwk"] = float(cohen_kappa_score(y_true, y_pred, labels=CLASS_LABELS, weights="quadratic"))
    except Exception:
        out["qwk"] = np.nan

    out["ordinal_mae"] = ordinal_mae(y_true, y_pred)
    out["ordinal_rmse"] = ordinal_rmse(y_true, y_pred)
    out["adjacent_accuracy"] = adjacent_accuracy(y_true, y_pred)

    try:
        out["log_loss"] = float(log_loss(y_true, proba, labels=CLASS_LABELS))
    except Exception:
        out["log_loss"] = np.nan

    out["brier"] = multiclass_brier(y_true, proba)
    out["ece"] = expected_calibration_error(y_true, proba)

    expected_class = expected_class_from_proba(proba)
    out["expected_class_mae"] = float(np.mean(np.abs(y_true - expected_class)))
    out["expected_class_rmse"] = float(np.sqrt(np.mean((y_true - expected_class) ** 2)))

    ent = entropy_from_proba(proba)
    out["mean_entropy"] = float(np.mean(ent))
    out["mean_normalized_entropy"] = float(np.mean(ent / np.log(len(CLASS_LABELS))))
    out["mean_max_probability"] = float(np.mean(np.max(proba, axis=1)))
    out["mean_probability_margin"] = float(np.mean(np.sort(proba, axis=1)[:, -1] - np.sort(proba, axis=1)[:, -2]))
    out["mean_ordinal_variance"] = float(np.mean(ordinal_variance_from_proba(proba)))

    return out

def score_for_selection(metrics):
    score = 0.0

    for metric, weight in SELECTION_METRIC_WEIGHTS.items():
        if metric == "neg_ordinal_mae":
            value = -float(metrics.get("ordinal_mae", np.nan))
        else:
            value = float(metrics.get(metric, np.nan))

        if not np.isfinite(value):
            value = -1e9

        score += weight * value

    return float(score)

def make_purged_walk_forward_splits(index, horizon, fold_specs=WALK_FORWARD_FOLDS):
    index = pd.DatetimeIndex(index).sort_values()
    n = len(index)

    splits = []

    for spec in fold_specs:
        fold_id = spec["fold_id"]

        train_end = int(np.floor(n * spec["train_end_frac"]))
        val_end = int(np.floor(n * spec["val_end_frac"]))
        test_end = int(np.floor(n * spec["test_end_frac"]))

        train_start = 0
        val_start = min(train_end + horizon, n)
        test_start = min(val_end + horizon, n)

        train_idx = index[train_start:train_end]
        val_idx = index[val_start:val_end]
        test_idx = index[test_start:test_end]

        if len(train_idx) == 0 or len(val_idx) == 0 or len(test_idx) == 0:
            print(f"Warning: empty split in {fold_id} for horizon {horizon}.")
            continue

        splits.append({
            "fold_id": fold_id,
            "horizon": horizon,
            "embargo": horizon,
            "train_start": train_idx.min(),
            "train_end": train_idx.max(),
            "validation_start": val_idx.min(),
            "validation_end": val_idx.max(),
            "test_start": test_idx.min(),
            "test_end": test_idx.max(),
            "train_idx": train_idx,
            "validation_idx": val_idx,
            "test_idx": test_idx,
            "n_train": len(train_idx),
            "n_validation": len(val_idx),
            "n_test": len(test_idx),
        })

    return splits

def save_table(df, local_name, global_name):
    local_path = TABLE_RUN_DIR / local_name
    global_path = TABLE_DIR / global_name

    df.to_csv(local_path, index=False)
    df.to_csv(global_path, index=False)

    return local_path, global_path

# ============================================================
# 4. Load leakage-controlled AURORA modeling dataset
# ============================================================

print("\n" + "=" * 80)
print("Step 1: Loading leakage-controlled AURORA modeling dataset")
print("=" * 80)

if AURORA_FEATURE_PATH.exists():
    df = pd.read_parquet(AURORA_FEATURE_PATH)
else:
    df = pd.read_csv(AURORA_FEATURE_CSV_PATH)

if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
else:
    df.index = pd.to_datetime(df.index)

df = df.sort_index()
df.index.name = "date"

print("Dataset shape:", df.shape)
print("Date range   :", df.index.min().date(), "to", df.index.max().date())

for target in TARGET_COLS:
    if target not in df.columns:
        raise ValueError(f"Target column missing: {target}")

target_set = set(TARGET_COLS)

feature_cols = [
    c for c in df.columns
    if c not in target_set
    and not c.lower().startswith("target")
    and not c.lower().endswith("_target")
]

# Keep numeric features only.
numeric_feature_cols = []
for c in feature_cols:
    if pd.api.types.is_numeric_dtype(df[c]):
        numeric_feature_cols.append(c)

feature_cols = numeric_feature_cols

if len(feature_cols) == 0:
    raise ValueError("No numeric feature columns found.")

print("Number of feature columns:", len(feature_cols))
print("Targets:", TARGET_COLS)

# Drop rows with missing target values for each target later.
dataset_profile_rows = []
for target in TARGET_COLS:
    tmp = df.dropna(subset=[target]).copy()
    dist = tmp[target].astype(int).value_counts(normalize=False).sort_index()
    dist_pct = tmp[target].astype(int).value_counts(normalize=True).sort_index()

    for cls in CLASS_LABELS:
        dataset_profile_rows.append({
            "target_col": target,
            "class": cls,
            "regime_label": REGIME_LABELS[cls],
            "count": int(dist.get(cls, 0)),
            "proportion": float(dist_pct.get(cls, 0.0)),
        })

dataset_profile_df = pd.DataFrame(dataset_profile_rows)

save_table(
    dataset_profile_df,
    local_name="roma_dataset_target_distribution.csv",
    global_name=f"table_R1_01_roma_dataset_target_distribution_{RUN_ID}.csv",
)

print("\nTarget distributions:")
print(dataset_profile_df.to_string(index=False))

# ============================================================
# 5. Define ROMA model zoo
# ============================================================

print("\n" + "=" * 80)
print("Step 2: Defining ROMA model zoo")
print("=" * 80)

def make_model_zoo():
    models = {}

    models["ROMA_M0_dummy_prior"] = DummyClassifier(
        strategy="prior",
        random_state=RANDOM_STATE,
    )

    models["ROMA_M1_logistic_balanced"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            multi_class="auto",
            class_weight="balanced",
            max_iter=2000,
            C=0.75,
            solver="lbfgs",
            random_state=RANDOM_STATE,
        )),
    ])

    models["ROMA_M2_random_forest_balanced"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=500,
            max_depth=5,
            min_samples_leaf=10,
            max_features="sqrt",
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        )),
    ])

    models["ROMA_M3_extra_trees_balanced"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            n_estimators=700,
            max_depth=6,
            min_samples_leaf=8,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        )),
    ])

    models["ROMA_M4_hist_gradient_boosting"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(
            max_iter=250,
            learning_rate=0.04,
            max_leaf_nodes=15,
            l2_regularization=0.10,
            random_state=RANDOM_STATE,
        )),
    ])

    if HAS_LIGHTGBM:
        models["ROMA_M5_lightgbm_multiclass"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", lgb.LGBMClassifier(
                objective="multiclass",
                num_class=len(CLASS_LABELS),
                n_estimators=500,
                learning_rate=0.03,
                max_depth=4,
                num_leaves=15,
                min_child_samples=25,
                subsample=0.85,
                colsample_bytree=0.75,
                reg_alpha=0.10,
                reg_lambda=0.50,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
                verbosity=-1,
            )),
        ])

    if HAS_XGBOOST:
        models["ROMA_M6_xgboost_multiclass"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                objective="multi:softprob",
                num_class=len(CLASS_LABELS),
                n_estimators=400,
                learning_rate=0.03,
                max_depth=3,
                min_child_weight=3.0,
                subsample=0.85,
                colsample_bytree=0.75,
                reg_lambda=1.00,
                reg_alpha=0.10,
                eval_metric="mlogloss",
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
                verbosity=0,
            )),
        ])

    return models

MODEL_ZOO = make_model_zoo()

model_catalog_df = pd.DataFrame([
    {
        "model_name": name,
        "model_class": str(model),
        "purpose": (
            "ROMA return-seeking regime probability source. "
            "Final allocation comparison will occur in R2 and R3."
        ),
    }
    for name, model in MODEL_ZOO.items()
])

save_table(
    model_catalog_df,
    local_name="roma_model_catalog.csv",
    global_name=f"table_R1_02_roma_model_catalog_{RUN_ID}.csv",
)

print("Models:")
for name in MODEL_ZOO:
    print(" -", name)

# ============================================================
# 6. Main training loop
# ============================================================

print("\n" + "=" * 80)
print("Step 3: Training ROMA models under purged walk-forward splits")
print("=" * 80)

all_metric_rows = []
all_prediction_frames = []
all_probability_frames = []
split_report_rows = []
ensemble_weight_rows = []

for target_col in TARGET_COLS:
    horizon = TARGET_HORIZON_MAP[target_col]

    print("\n" + "#" * 80)
    print("Target:", target_col, "| Horizon:", horizon)
    print("#" * 80)

    target_df = df.dropna(subset=[target_col]).copy()
    target_df[target_col] = target_df[target_col].astype(int)

    X_all = target_df[feature_cols].copy()
    y_all = target_df[target_col].copy()

    splits = make_purged_walk_forward_splits(
        index=target_df.index,
        horizon=horizon,
        fold_specs=WALK_FORWARD_FOLDS,
    )

    for split in splits:
        fold_id = split["fold_id"]

        print("\n" + "-" * 80)
        print(f"{target_col} | {fold_id}")
        print("-" * 80)
        print(
            "Train:",
            split["n_train"],
            split["train_start"].date(),
            "to",
            split["train_end"].date(),
        )
        print(
            "Validation:",
            split["n_validation"],
            split["validation_start"].date(),
            "to",
            split["validation_end"].date(),
        )
        print(
            "Test:",
            split["n_test"],
            split["test_start"].date(),
            "to",
            split["test_end"].date(),
        )
        print("Embargo:", split["embargo"])

        split_report_rows.append({
            "target_col": target_col,
            "horizon": horizon,
            "fold_id": fold_id,
            "embargo": split["embargo"],
            "train_start": split["train_start"],
            "train_end": split["train_end"],
            "validation_start": split["validation_start"],
            "validation_end": split["validation_end"],
            "test_start": split["test_start"],
            "test_end": split["test_end"],
            "n_train": split["n_train"],
            "n_validation": split["n_validation"],
            "n_test": split["n_test"],
        })

        train_idx = split["train_idx"]
        val_idx = split["validation_idx"]
        test_idx = split["test_idx"]

        X_train = X_all.loc[train_idx]
        y_train = y_all.loc[train_idx]

        X_val = X_all.loc[val_idx]
        y_val = y_all.loc[val_idx]

        X_test = X_all.loc[test_idx]
        y_test = y_all.loc[test_idx]

        trained_models = {}
        val_proba_by_model = {}
        test_proba_by_model = {}
        validation_selection_scores = {}

        for model_name, model_template in MODEL_ZOO.items():
            print("Training:", model_name)

            try:
                model = clone(model_template)

                # XGBoost can fail if labels missing; classes are still 0..4 in global target,
                # but some folds may miss rare classes. The fixed proba helper handles output columns.
                model.fit(X_train, y_train)

                trained_models[model_name] = model

                val_proba = predict_proba_fixed_classes(model, X_val)
                test_proba = predict_proba_fixed_classes(model, X_test)

                val_proba_by_model[model_name] = val_proba
                test_proba_by_model[model_name] = test_proba

                for split_name, X_split, y_split, proba_split in [
                    ("validation", X_val, y_val, val_proba),
                    ("test", X_test, y_test, test_proba),
                ]:
                    metrics = calculate_metrics(y_split.values, proba_split)
                    selection_score = score_for_selection(metrics) if split_name == "validation" else np.nan

                    row = {
                        "run_id": RUN_ID,
                        "project_code": PROJECT_CODE,
                        "target_col": target_col,
                        "horizon": horizon,
                        "fold_id": fold_id,
                        "split": split_name,
                        "model_name": model_name,
                        "selection_score": selection_score,
                        **metrics,
                    }
                    all_metric_rows.append(row)

                    if split_name == "validation":
                        validation_selection_scores[model_name] = selection_score

                    pred = hard_prediction_from_proba(proba_split)
                    exp_cls = expected_class_from_proba(proba_split)
                    ord_var = ordinal_variance_from_proba(proba_split)
                    ent = entropy_from_proba(proba_split)
                    norm_ent = ent / np.log(len(CLASS_LABELS))
                    confidence = 1.0 - norm_ent
                    margin = np.sort(proba_split, axis=1)[:, -1] - np.sort(proba_split, axis=1)[:, -2]

                    split_index = y_split.index

                    pred_df = pd.DataFrame({
                        "date": split_index,
                        "run_id": RUN_ID,
                        "project_code": PROJECT_CODE,
                        "target_col": target_col,
                        "horizon": horizon,
                        "fold_id": fold_id,
                        "split": split_name,
                        "model_name": model_name,
                        "y_true": y_split.values.astype(int),
                        "y_pred": pred.astype(int),
                        "expected_class": exp_cls,
                        "ordinal_variance": ord_var,
                        "entropy": ent,
                        "normalized_entropy": norm_ent,
                        "confidence_score": confidence,
                        "probability_margin": margin,
                    })

                    proba_df = pred_df.copy()
                    for j, cls in enumerate(CLASS_LABELS):
                        proba_df[f"proba_class_{cls}"] = proba_split[:, j]

                    all_prediction_frames.append(pred_df)
                    all_probability_frames.append(proba_df)

            except Exception as e:
                print("  FAILED:", model_name, repr(e))

                for split_name in ["validation", "test"]:
                    fail_row = {
                        "run_id": RUN_ID,
                        "project_code": PROJECT_CODE,
                        "target_col": target_col,
                        "horizon": horizon,
                        "fold_id": fold_id,
                        "split": split_name,
                        "model_name": model_name,
                        "selection_score": np.nan,
                        "accuracy": np.nan,
                        "balanced_accuracy": np.nan,
                        "macro_f1": np.nan,
                        "weighted_f1": np.nan,
                        "qwk": np.nan,
                        "ordinal_mae": np.nan,
                        "ordinal_rmse": np.nan,
                        "adjacent_accuracy": np.nan,
                        "log_loss": np.nan,
                        "brier": np.nan,
                        "ece": np.nan,
                        "expected_class_mae": np.nan,
                        "expected_class_rmse": np.nan,
                        "mean_entropy": np.nan,
                        "mean_normalized_entropy": np.nan,
                        "mean_max_probability": np.nan,
                        "mean_probability_margin": np.nan,
                        "mean_ordinal_variance": np.nan,
                        "error": repr(e),
                    }
                    all_metric_rows.append(fail_row)

        # ----------------------------------------------------
        # Fold-level ROMA validation-weighted probability ensemble
        # ----------------------------------------------------
        valid_scores = {
            k: v for k, v in validation_selection_scores.items()
            if np.isfinite(v) and k in val_proba_by_model and k in test_proba_by_model
        }

        if len(valid_scores) > 0:
            score_series = pd.Series(valid_scores).sort_values(ascending=False)
            top_models = score_series.head(min(ENSEMBLE_TOP_K, len(score_series)))

            shifted = top_models - top_models.min()
            raw_weights = shifted + ENSEMBLE_MIN_WEIGHT

            if raw_weights.sum() <= 0:
                weights = pd.Series(1.0 / len(top_models), index=top_models.index)
            else:
                weights = raw_weights / raw_weights.sum()

            ensemble_name = "ROMA_E1_validation_weighted_probability_ensemble"

            for model_name, weight in weights.items():
                ensemble_weight_rows.append({
                    "run_id": RUN_ID,
                    "target_col": target_col,
                    "horizon": horizon,
                    "fold_id": fold_id,
                    "ensemble_name": ensemble_name,
                    "member_model": model_name,
                    "validation_selection_score": float(score_series[model_name]),
                    "ensemble_weight": float(weight),
                })

            for split_name, y_split, proba_dict in [
                ("validation", y_val, val_proba_by_model),
                ("test", y_test, test_proba_by_model),
            ]:
                ensemble_proba = None

                for model_name, weight in weights.items():
                    if ensemble_proba is None:
                        ensemble_proba = weight * proba_dict[model_name]
                    else:
                        ensemble_proba += weight * proba_dict[model_name]

                ensemble_proba = normalize_proba_array(ensemble_proba)

                metrics = calculate_metrics(y_split.values, ensemble_proba)
                selection_score = score_for_selection(metrics) if split_name == "validation" else np.nan

                all_metric_rows.append({
                    "run_id": RUN_ID,
                    "project_code": PROJECT_CODE,
                    "target_col": target_col,
                    "horizon": horizon,
                    "fold_id": fold_id,
                    "split": split_name,
                    "model_name": ensemble_name,
                    "selection_score": selection_score,
                    **metrics,
                })

                pred = hard_prediction_from_proba(ensemble_proba)
                exp_cls = expected_class_from_proba(ensemble_proba)
                ord_var = ordinal_variance_from_proba(ensemble_proba)
                ent = entropy_from_proba(ensemble_proba)
                norm_ent = ent / np.log(len(CLASS_LABELS))
                confidence = 1.0 - norm_ent
                margin = np.sort(ensemble_proba, axis=1)[:, -1] - np.sort(ensemble_proba, axis=1)[:, -2]

                pred_df = pd.DataFrame({
                    "date": y_split.index,
                    "run_id": RUN_ID,
                    "project_code": PROJECT_CODE,
                    "target_col": target_col,
                    "horizon": horizon,
                    "fold_id": fold_id,
                    "split": split_name,
                    "model_name": ensemble_name,
                    "y_true": y_split.values.astype(int),
                    "y_pred": pred.astype(int),
                    "expected_class": exp_cls,
                    "ordinal_variance": ord_var,
                    "entropy": ent,
                    "normalized_entropy": norm_ent,
                    "confidence_score": confidence,
                    "probability_margin": margin,
                })

                proba_df = pred_df.copy()
                for j, cls in enumerate(CLASS_LABELS):
                    proba_df[f"proba_class_{cls}"] = ensemble_proba[:, j]

                all_prediction_frames.append(pred_df)
                all_probability_frames.append(proba_df)

            print("Created ensemble:", ensemble_name)
            print("Ensemble weights:")
            print(weights.to_string())

# ============================================================
# 7. Save aggregate raw outputs
# ============================================================

print("\n" + "=" * 80)
print("Step 4: Saving aggregate ROMA metrics, probabilities, and predictions")
print("=" * 80)

metrics_df = pd.DataFrame(all_metric_rows)
predictions_df = pd.concat(all_prediction_frames, axis=0, ignore_index=True)
probabilities_df = pd.concat(all_probability_frames, axis=0, ignore_index=True)

split_report_df = pd.DataFrame(split_report_rows)
ensemble_weights_df = pd.DataFrame(ensemble_weight_rows)

for dfx in [predictions_df, probabilities_df]:
    dfx["date"] = pd.to_datetime(dfx["date"])
    dfx.sort_values(["target_col", "fold_id", "split", "model_name", "date"], inplace=True)

metrics_df.to_csv(METRIC_DIR / "roma_all_metrics.csv", index=False)
metrics_df.to_parquet(METRIC_DIR / "roma_all_metrics.parquet", index=False)

metrics_df.to_csv(TABLE_DIR / f"table_R1_03_roma_all_metrics_{RUN_ID}.csv", index=False)

predictions_df.to_csv(PRED_DIR / "roma_all_predictions.csv", index=False)
predictions_df.to_parquet(PRED_DIR / "roma_all_predictions.parquet", index=False)

probabilities_df.to_csv(PROBA_DIR / "roma_all_probabilities.csv", index=False)
probabilities_df.to_parquet(PROBA_DIR / "roma_all_probabilities.parquet", index=False)

split_report_df.to_csv(TABLE_RUN_DIR / "roma_purged_walk_forward_split_report.csv", index=False)
split_report_df.to_csv(TABLE_DIR / f"table_R1_04_roma_purged_walk_forward_split_report_{RUN_ID}.csv", index=False)

ensemble_weights_df.to_csv(TABLE_RUN_DIR / "roma_ensemble_weights.csv", index=False)
ensemble_weights_df.to_csv(TABLE_DIR / f"table_R1_05_roma_ensemble_weights_{RUN_ID}.csv", index=False)

print("Metrics shape      :", metrics_df.shape)
print("Predictions shape  :", predictions_df.shape)
print("Probabilities shape:", probabilities_df.shape)
print("Split report shape :", split_report_df.shape)
print("Ensemble weights   :", ensemble_weights_df.shape)

# ============================================================
# 8. Build aggregate validation/test leaderboards
# ============================================================

print("\n" + "=" * 80)
print("Step 5: Building ROMA validation and test leaderboards")
print("=" * 80)

metric_cols_for_mean = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "qwk",
    "ordinal_mae",
    "ordinal_rmse",
    "adjacent_accuracy",
    "log_loss",
    "brier",
    "ece",
    "expected_class_mae",
    "expected_class_rmse",
    "mean_entropy",
    "mean_normalized_entropy",
    "mean_max_probability",
    "mean_probability_margin",
    "mean_ordinal_variance",
    "selection_score",
]

agg_metric_df = (
    metrics_df
    .groupby(["target_col", "horizon", "split", "model_name"], as_index=False)[metric_cols_for_mean]
    .mean(numeric_only=True)
)

# Composite rank for validation and test.
leaderboard_frames = []

for (target_col, split_name), g in agg_metric_df.groupby(["target_col", "split"]):
    tmp = g.copy()

    tmp["rank_macro_f1"] = tmp["macro_f1"].rank(ascending=False, method="min")
    tmp["rank_balanced_accuracy"] = tmp["balanced_accuracy"].rank(ascending=False, method="min")
    tmp["rank_qwk"] = tmp["qwk"].rank(ascending=False, method="min")
    tmp["rank_ordinal_mae"] = tmp["ordinal_mae"].rank(ascending=True, method="min")
    tmp["rank_ece"] = tmp["ece"].rank(ascending=True, method="min")

    tmp["composite_rank"] = (
        tmp["rank_macro_f1"]
        + tmp["rank_balanced_accuracy"]
        + tmp["rank_qwk"]
        + tmp["rank_ordinal_mae"]
        + tmp["rank_ece"]
    ) / 5.0

    tmp = tmp.sort_values(
        ["composite_rank", "macro_f1", "balanced_accuracy", "qwk"],
        ascending=[True, False, False, False],
    )

    leaderboard_frames.append(tmp)

leaderboard_df = pd.concat(leaderboard_frames, axis=0, ignore_index=True)

leaderboard_df.to_csv(TABLE_RUN_DIR / "roma_aggregate_leaderboard.csv", index=False)
leaderboard_df.to_csv(TABLE_DIR / f"table_R1_06_roma_aggregate_leaderboard_{RUN_ID}.csv", index=False)

print("Aggregate leaderboard:")
print(
    leaderboard_df[
        [
            "target_col",
            "split",
            "model_name",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "qwk",
            "ordinal_mae",
            "ece",
            "selection_score",
            "composite_rank",
        ]
    ].to_string(index=False)
)

# ============================================================
# 9. Select final ROMA probability sources for R2
# ============================================================

print("\n" + "=" * 80)
print("Step 6: Selecting validation-supported ROMA probability sources for R2")
print("=" * 80)

selected_rows = []

for target_col in TARGET_COLS:
    val_lb = leaderboard_df[
        (leaderboard_df["target_col"] == target_col)
        & (leaderboard_df["split"] == "validation")
    ].copy()

    if val_lb.empty:
        raise ValueError(f"No validation leaderboard for {target_col}")

    # Primary selection rule:
    # choose best validation composite rank.
    selected = val_lb.sort_values(
        ["composite_rank", "selection_score", "macro_f1", "balanced_accuracy", "qwk"],
        ascending=[True, False, False, False, False],
    ).iloc[0]

    selected_rows.append({
        "run_id": RUN_ID,
        "project_code": PROJECT_CODE,
        "target_col": target_col,
        "horizon": int(TARGET_HORIZON_MAP[target_col]),
        "selected_model_name": selected["model_name"],
        "selection_rule": "best_validation_aggregate_composite_rank",
        "validation_composite_rank": float(selected["composite_rank"]),
        "validation_selection_score": float(selected["selection_score"]) if pd.notna(selected["selection_score"]) else np.nan,
        "validation_macro_f1": float(selected["macro_f1"]),
        "validation_balanced_accuracy": float(selected["balanced_accuracy"]),
        "validation_qwk": float(selected["qwk"]),
        "validation_ordinal_mae": float(selected["ordinal_mae"]),
        "validation_ece": float(selected["ece"]),
    })

selected_df = pd.DataFrame(selected_rows)

selected_df.to_csv(TABLE_RUN_DIR / "roma_selected_probability_sources.csv", index=False)
selected_df.to_csv(TABLE_DIR / f"table_R1_07_roma_selected_probability_sources_{RUN_ID}.csv", index=False)

print("Selected probability sources:")
print(selected_df.to_string(index=False))

# ============================================================
# 10. Export selected probabilities and predictions
# ============================================================

print("\n" + "=" * 80)
print("Step 7: Exporting selected ROMA probabilities and R2 input index")
print("=" * 80)

r2_index_rows = []

for _, row in selected_df.iterrows():
    target_col = row["target_col"]
    horizon = int(row["horizon"])
    selected_model = row["selected_model_name"]

    selected_proba = probabilities_df[
        (probabilities_df["target_col"] == target_col)
        & (probabilities_df["model_name"] == selected_model)
        & (probabilities_df["split"].isin(["validation", "test"]))
    ].copy()

    selected_pred = predictions_df[
        (predictions_df["target_col"] == target_col)
        & (predictions_df["model_name"] == selected_model)
        & (predictions_df["split"].isin(["validation", "test"]))
    ].copy()

    if selected_proba.empty:
        raise ValueError(f"No selected probabilities for {target_col} | {selected_model}")

    selected_proba = selected_proba.sort_values(["date", "fold_id", "split"])
    selected_pred = selected_pred.sort_values(["date", "fold_id", "split"])

    base = f"selected_ROMA_probabilities_{safe_name(target_col)}_{safe_name(selected_model)}"
    proba_parquet = PROBA_DIR / f"{base}.parquet"
    proba_csv = PROBA_DIR / f"{base}.csv"

    selected_proba.to_parquet(proba_parquet, index=False)
    selected_proba.to_csv(proba_csv, index=False)

    pred_base = f"selected_ROMA_predictions_{safe_name(target_col)}_{safe_name(selected_model)}"
    pred_parquet = PRED_DIR / f"{pred_base}.parquet"
    pred_csv = PRED_DIR / f"{pred_base}.csv"

    selected_pred.to_parquet(pred_parquet, index=False)
    selected_pred.to_csv(pred_csv, index=False)

    r2_index_rows.append({
        "run_id": RUN_ID,
        "project_code": PROJECT_CODE,
        "target_col": target_col,
        "horizon": horizon,
        "selected_model_name": selected_model,
        "probability_path_parquet": str(proba_parquet),
        "probability_path_csv": str(proba_csv),
        "prediction_path_parquet": str(pred_parquet),
        "prediction_path_csv": str(pred_csv),
        "selection_rule": row["selection_rule"],
        "validation_composite_rank": row["validation_composite_rank"],
        "validation_macro_f1": row["validation_macro_f1"],
        "validation_balanced_accuracy": row["validation_balanced_accuracy"],
        "validation_qwk": row["validation_qwk"],
        "validation_ordinal_mae": row["validation_ordinal_mae"],
        "validation_ece": row["validation_ece"],
    })

r2_index_df = pd.DataFrame(r2_index_rows)

r2_index_path = RUN_ROOT / "NOTEBOOK_R2_ROMA_INPUT_INDEX.csv"
r2_index_global_path = TABLE_DIR / f"table_R1_08_NOTEBOOK_R2_ROMA_INPUT_INDEX_{RUN_ID}.csv"

r2_index_df.to_csv(r2_index_path, index=False)
r2_index_df.to_csv(r2_index_global_path, index=False)

print("R2 input index:")
print(r2_index_df.to_string(index=False))
print("Saved:", r2_index_path)

# ============================================================
# 11. Export latest-fold deduplicated selected probabilities
# ============================================================

print("\n" + "=" * 80)
print("Step 8: Exporting latest-fold deduplicated selected probabilities")
print("=" * 80)

def latest_fold_deduplicate(df_in, split_filter=None):
    df2 = df_in.copy()
    df2["date"] = pd.to_datetime(df2["date"])

    if split_filter is not None:
        if isinstance(split_filter, str):
            split_filter = [split_filter]
        df2 = df2[df2["split"].isin(split_filter)].copy()

    if df2.empty:
        return df2

    df2["fold_number"] = (
        df2["fold_id"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .fillna("0")
        .astype(int)
    )

    split_priority = {"train": 0, "validation": 1, "test": 2}
    df2["split_priority"] = df2["split"].map(split_priority).fillna(0).astype(int)

    df2 = df2.sort_values(["date", "split_priority", "fold_number"])
    df2 = df2.drop_duplicates(subset=["date"], keep="last")
    df2 = df2.drop(columns=["fold_number", "split_priority"], errors="ignore")
    df2 = df2.sort_values("date")

    return df2

dedup_index_rows = []

for _, row in r2_index_df.iterrows():
    target_col = row["target_col"]
    horizon = int(row["horizon"])
    model_name = row["selected_model_name"]

    selected_proba = pd.read_parquet(row["probability_path_parquet"])

    for split_filter_name, split_filter in [
        ("oos_validation_plus_test", ["validation", "test"]),
        ("strict_test_only", ["test"]),
    ]:
        dedup = latest_fold_deduplicate(selected_proba, split_filter=split_filter)

        base = f"dedup_{split_filter_name}_ROMA_probabilities_{safe_name(target_col)}_{safe_name(model_name)}"
        out_parquet = PROBA_DIR / f"{base}.parquet"
        out_csv = PROBA_DIR / f"{base}.csv"

        dedup.to_parquet(out_parquet, index=False)
        dedup.to_csv(out_csv, index=False)

        if len(dedup) > 0:
            start_date = dedup["date"].min()
            end_date = dedup["date"].max()
        else:
            start_date = pd.NaT
            end_date = pd.NaT

        dedup_index_rows.append({
            "run_id": RUN_ID,
            "target_col": target_col,
            "horizon": horizon,
            "selected_model_name": model_name,
            "dedup_split": split_filter_name,
            "n_rows": int(len(dedup)),
            "start_date": start_date,
            "end_date": end_date,
            "probability_path_parquet": str(out_parquet),
            "probability_path_csv": str(out_csv),
        })

dedup_index_df = pd.DataFrame(dedup_index_rows)

dedup_index_path = RUN_ROOT / "NOTEBOOK_R2_ROMA_DEDUP_INPUT_INDEX.csv"
dedup_index_global_path = TABLE_DIR / f"table_R1_09_NOTEBOOK_R2_ROMA_DEDUP_INPUT_INDEX_{RUN_ID}.csv"

dedup_index_df.to_csv(dedup_index_path, index=False)
dedup_index_df.to_csv(dedup_index_global_path, index=False)

print("Deduplicated R2 input index:")
print(dedup_index_df.to_string(index=False))

# ============================================================
# 12. Diagnostic summaries
# ============================================================

print("\n" + "=" * 80)
print("Step 9: Creating diagnostic summaries")
print("=" * 80)

test_leaderboard = leaderboard_df[leaderboard_df["split"] == "test"].copy()
validation_leaderboard = leaderboard_df[leaderboard_df["split"] == "validation"].copy()

diagnostic_rows = []

for target_col in TARGET_COLS:
    selected_model = selected_df[selected_df["target_col"] == target_col]["selected_model_name"].iloc[0]

    selected_val = validation_leaderboard[
        (validation_leaderboard["target_col"] == target_col)
        & (validation_leaderboard["model_name"] == selected_model)
    ].iloc[0]

    selected_test = test_leaderboard[
        (test_leaderboard["target_col"] == target_col)
        & (test_leaderboard["model_name"] == selected_model)
    ].iloc[0]

    best_test = test_leaderboard[
        test_leaderboard["target_col"] == target_col
    ].sort_values("composite_rank").iloc[0]

    diagnostic_rows.append({
        "target_col": target_col,
        "selected_model": selected_model,
        "validation_composite_rank": selected_val["composite_rank"],
        "validation_macro_f1": selected_val["macro_f1"],
        "validation_balanced_accuracy": selected_val["balanced_accuracy"],
        "validation_qwk": selected_val["qwk"],
        "test_composite_rank_of_selected_model": selected_test["composite_rank"],
        "test_macro_f1_of_selected_model": selected_test["macro_f1"],
        "test_balanced_accuracy_of_selected_model": selected_test["balanced_accuracy"],
        "test_qwk_of_selected_model": selected_test["qwk"],
        "best_test_model_for_diagnostic_only": best_test["model_name"],
        "best_test_composite_rank_for_diagnostic_only": best_test["composite_rank"],
        "important_note": (
            "Selected model is chosen by validation aggregate ranking. "
            "Best test model is reported only as diagnostic and must not be used "
            "for final model selection."
        ),
    })

diagnostic_df = pd.DataFrame(diagnostic_rows)

diagnostic_df.to_csv(TABLE_RUN_DIR / "roma_R1_diagnostic_summary.csv", index=False)
diagnostic_df.to_csv(TABLE_DIR / f"table_R1_10_roma_R1_diagnostic_summary_{RUN_ID}.csv", index=False)

print("Diagnostic summary:")
print(diagnostic_df.to_string(index=False))

# ============================================================
# 13. Validation report and manifest
# ============================================================

print("\n" + "=" * 80)
print("Step 10: Saving validation report and SHA256 manifest")
print("=" * 80)

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "R1_ROMA_rebuild_regime_signals_leakage_controlled.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "purpose": (
        "Clean rebuild of ROMA regime probability signals using the leakage-controlled "
        "AURORA feature-label dataset and purged walk-forward evaluation."
    ),
    "shared_aurora_feature_path": str(AURORA_FEATURE_PATH if AURORA_FEATURE_PATH.exists() else AURORA_FEATURE_CSV_PATH),
    "etf_universe": ETF_UNIVERSE,
    "cash_column": CASH_COL,
    "target_columns": TARGET_COLS,
    "target_horizon_map": TARGET_HORIZON_MAP,
    "class_labels": CLASS_LABELS,
    "regime_labels": REGIME_LABELS,
    "walk_forward_folds": WALK_FORWARD_FOLDS,
    "model_names": list(MODEL_ZOO.keys()) + ["ROMA_E1_validation_weighted_probability_ensemble"],
    "selection_metric_weights": SELECTION_METRIC_WEIGHTS,
    "selected_probability_sources": selected_df.to_dict(orient="records"),
    "deduplicated_input_index": str(dedup_index_path),
    "r2_input_index": str(r2_index_path),
    "important_methodological_notes": [
        "ROMA is rebuilt using the same leakage-controlled AURORA dataset.",
        "Purged walk-forward splits use target-specific embargoes.",
        "Selected probability sources are chosen by validation aggregate composite rank.",
        "Test-set best models are diagnostic only and must not be used for model selection.",
        "Final ROMA allocation and comparison against AURORA will be performed in R2/R3/Notebook13.",
    ],
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "metrics": str(METRIC_DIR),
        "probabilities": str(PROBA_DIR),
        "predictions": str(PRED_DIR),
        "tables": str(TABLE_RUN_DIR),
        "reports": str(REPORT_RUN_DIR),
    },
    "educational_note": (
        "This notebook performs research signal generation only and does not provide personalized financial advice."
    ),
}

validation_report_path = REPORT_RUN_DIR / "ROMA_R1_validation_report.json"
validation_report_global_path = REPORT_DIR / f"ROMA_R1_validation_report_{RUN_ID}.json"

save_json(validation_report_path, validation_report)
save_json(validation_report_global_path, validation_report)

manifest_df = make_file_manifest(RUN_ROOT)

manifest_path = REPORT_RUN_DIR / "ROMA_R1_file_manifest_SHA256.csv"
manifest_global_path = REPORT_DIR / f"ROMA_R1_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(manifest_global_path, index=False)

# ============================================================
# 14. Final summary
# ============================================================

print("\n" + "=" * 80)
print("ROMA-TWETF R1 COMPLETE")
print("=" * 80)
print("Run ID                         :", RUN_ID)
print("Run root                       :", RUN_ROOT)
print("Dataset target distribution    :", TABLE_DIR / f"table_R1_01_roma_dataset_target_distribution_{RUN_ID}.csv")
print("Model catalog                  :", TABLE_DIR / f"table_R1_02_roma_model_catalog_{RUN_ID}.csv")
print("All metrics                    :", TABLE_DIR / f"table_R1_03_roma_all_metrics_{RUN_ID}.csv")
print("Split report                   :", TABLE_DIR / f"table_R1_04_roma_purged_walk_forward_split_report_{RUN_ID}.csv")
print("Ensemble weights               :", TABLE_DIR / f"table_R1_05_roma_ensemble_weights_{RUN_ID}.csv")
print("Aggregate leaderboard          :", TABLE_DIR / f"table_R1_06_roma_aggregate_leaderboard_{RUN_ID}.csv")
print("Selected probability sources   :", TABLE_DIR / f"table_R1_07_roma_selected_probability_sources_{RUN_ID}.csv")
print("R2 input index                 :", r2_index_path)
print("R2 dedup input index           :", dedup_index_path)
print("Diagnostic summary             :", TABLE_DIR / f"table_R1_10_roma_R1_diagnostic_summary_{RUN_ID}.csv")
print("Probabilities directory        :", PROBA_DIR)
print("Predictions directory          :", PRED_DIR)
print("Validation report              :", validation_report_path)
print("Manifest                       :", manifest_path)
print("=" * 80)

print("\nRecommended next notebook:")
print("R2_ROMA_aligned_allocation_backtest.ipynb")
print("\nImportant for R2:")
print("Use this file:")
print(dedup_index_path)

Mounted at /content/drive
ROMA-TWETF R1: Clean Leakage-Controlled Regime Signal Rebuild
Timestamp UTC       : 2026-06-25T02:36:29Z
Run ID              : 20260625_023629
AURORA feature path : /content/drive/MyDrive/AURORA_TWETF/data/modeling/AURORA_TWETF_features_with_labels.parquet
ETF return panel    : /content/drive/MyDrive/AURORA_TWETF/data/panels/AURORA_etf_return_panel.parquet
Run root            : /content/drive/MyDrive/AURORA_TWETF/outputs/ROMA_TWETF/purged_walk_forward_regime_signals/run_20260625_023629

Step 1: Loading leakage-controlled AURORA modeling dataset
Dataset shape: (1262, 417)
Date range   : 2021-01-06 to 2026-03-25
Number of feature columns: 415
Targets: ['TAIEX_regime_fixed_20d', 'TAIEX_regime_fixed_60d']

Target distributions:
            target_col  class regime_label  count  proportion
TAIEX_regime_fixed_20d      0  strong_bear     40    0.031696
TAIEX_regime_fixed_20d      1         bear    196    0.155309
TAIEX_regime_fixed_20d      2      neutral    536    0